# Embeddings e Vector Stores

Modelos de linguagem trabalham com texto, mas internamente precisam representar palavras e frases como **números**. Essa representação numérica é chamada de **embedding**: um vetor de números que captura o significado semântico do texto.

Textos com significados parecidos geram vetores próximos no espaço. Isso permite medir a **similaridade** entre textos de forma matemática, o que é a base para sistemas de busca semântica e RAG (Retrieval-Augmented Generation).

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Gerando embeddings

Vamos usar o modelo `text-embedding-3-small` da OpenAI para gerar embeddings. Ele transforma qualquer texto em um vetor de 1536 dimensões.

In [2]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [3]:
vetor = embeddings.embed_query("O Brasil e o maior pais da America do Sul")

print(f"Dimensoes: {len(vetor)}")
print(f"Primeiros 5 valores: {vetor[:5]}")

Dimensoes: 1536
Primeiros 5 valores: [0.022121012210845947, -0.013751386664807796, 0.015119325369596481, 0.0608372837305069, 0.017396222800016403]


Cada texto vira um vetor com 1536 números. Esses números não têm significado individual, mas a **relação entre vetores** é o que importa.

## Similaridade entre textos

Para medir o quanto dois textos são parecidos, calculamos a **similaridade do cosseno** entre seus vetores. O valor varia de 0 (nenhuma relação) a 1 (idênticos em significado).

In [4]:
from scipy.spatial.distance import cosine

def similaridade(texto1: str, texto2: str) -> float:
    """Calcula a similaridade entre dois textos usando embeddings."""
    v1 = embeddings.embed_query(texto1)
    v2 = embeddings.embed_query(texto2)
    return 1 - cosine(v1, v2)

In [5]:
# Textos semanticamente similares
print(similaridade(
    "O gato dormiu no sofa",
    "O felino descansou no sofa"
))

0.8255239800848293


In [6]:
# Textos sobre o mesmo tema
print(similaridade(
    "A economia brasileira cresceu 3% no ultimo trimestre",
    "O PIB do Brasil teve alta de 3% no periodo"
))

0.7268681454454665


In [7]:
# Textos sem relacao
print(similaridade(
    "O gato dormiu no sofa",
    "A economia brasileira cresceu 3% no ultimo trimestre"
))

0.2104809771745284


Os resultados mostram o poder dos embeddings:

- **"O gato dormiu no sofá" vs "O felino descansou no sofá"** → ~0.83 de similaridade. São frases com o mesmo significado usando sinônimos, e o modelo captura isso.
- **"Economia brasileira cresceu 3%" vs "PIB do Brasil teve alta de 3%"** → ~0.73. Mesmo tema, linguagem diferente.
- **"O gato dormiu no sofá" vs "A economia brasileira cresceu"** → ~0.21. Assuntos completamente diferentes, similaridade baixa.

O modelo entende sinônimos, paráfrases e relações semânticas — não apenas palavras idênticas.

## Vector Store

Um **vector store** é um banco de dados especializado em armazenar e buscar vetores. Em vez de buscar por palavras-chave (como um banco tradicional), ele busca por **similaridade semântica**: dado um texto de consulta, retorna os textos mais parecidos.

Vamos usar o `InMemoryVectorStore` do LangChain, que armazena tudo na memória RAM. É o mais simples para aprender e prototipar.

In [8]:
from langchain_core.vectorstores import InMemoryVectorStore

vectorstore = InMemoryVectorStore(embedding=embeddings)

## Adicionando documentos

Documentos no LangChain são representados pelo objeto `Document`, que tem dois campos: `page_content` (o texto) e `metadata` (informações extras como fonte, página, etc.).

In [9]:
from langchain_core.documents import Document

documentos = [
    Document(
        page_content="Python e uma linguagem de programacao muito usada em ciencia de dados e inteligencia artificial.",
        metadata={"categoria": "tecnologia"}
    ),
    Document(
        page_content="JavaScript e a principal linguagem para desenvolvimento web front-end.",
        metadata={"categoria": "tecnologia"}
    ),
    Document(
        page_content="O PIB do Brasil cresceu 3.4% em 2024, impulsionado pelo setor de servicos.",
        metadata={"categoria": "economia"}
    ),
    Document(
        page_content="A taxa Selic esta em 14.25% ao ano, definida pelo Banco Central do Brasil.",
        metadata={"categoria": "economia"}
    ),
    Document(
        page_content="Machine learning e uma subarea da inteligencia artificial que permite que sistemas aprendam a partir de dados.",
        metadata={"categoria": "tecnologia"}
    ),
]

vectorstore.add_documents(documentos)

['834df73a-e69e-48e1-a3f8-255b82c66426',
 '84d78ff6-aa26-481c-936e-90f5cc8feb7e',
 'd6635116-00de-4c48-9136-d554d5b7e317',
 '225d8b4f-2adc-4f94-ab2b-999d2931d7dd',
 '7da60e5a-257d-4f36-967e-c0474bcd5389']

## Busca por similaridade

Agora podemos buscar documentos por significado. O vector store converte a consulta em um vetor, compara com todos os vetores armazenados e retorna os mais próximos.

In [10]:
resultados = vectorstore.similarity_search("linguagem para IA", k=2)

for doc in resultados:
    print(f"[{doc.metadata['categoria']}] {doc.page_content}")

[tecnologia] Python e uma linguagem de programacao muito usada em ciencia de dados e inteligencia artificial.
[tecnologia] JavaScript e a principal linguagem para desenvolvimento web front-end.


In [11]:
resultados = vectorstore.similarity_search("juros e inflacao", k=2)

for doc in resultados:
    print(f"[{doc.metadata['categoria']}] {doc.page_content}")

[economia] A taxa Selic esta em 14.25% ao ano, definida pelo Banco Central do Brasil.
[economia] O PIB do Brasil cresceu 3.4% em 2024, impulsionado pelo setor de servicos.


A busca por "linguagem para IA" retornou documentos sobre Python e JavaScript — mesmo sem usar essas palavras exatas. E a busca por "juros e inflação" encontrou o documento sobre a Selic. O vector store entende o **significado**, não apenas palavras-chave.

Esse mecanismo é a base do RAG: em vez de alimentar o modelo com todos os documentos, buscamos apenas os relevantes para a pergunta do usuário e passamos como contexto. No próximo notebook, vamos implementar esse fluxo completo.